# Phase 5 — Final Recommendation Evaluation

This notebook evaluates the frozen Phase 4 routed architecture. It calculates ranking metrics and adds diversity, novelty, personalization, route, subgroup, uncertainty, failure-case and report-ready evaluation. It never tunes or replaces the selected model.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Configure Phase 5 inputs and constants


In [2]:
from pathlib import Path
import json
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as pds

BENCHMARK_ROOT = Path(
    '/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs'
)
ASSIST_ROOT = BENCHMARK_ROOT / 'assistments'
BASELINE_ROOT = Path(
    '/content/drive/MyDrive/datasets/recommendation_baseline_outputs'
)
PHASE4_ROOT = Path(
    '/content/drive/MyDrive/datasets/hybrid_recommender_gated_final_outputs'
)
OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/datasets/recommendation_evaluation_final_outputs'
)
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

PRODUCTION_MODEL = 'gated_cf_content_with_sequential_fallback'
PRIMARY_COMPARATOR = 'learner_neighbor_cf'
RANDOM_STATE = 42
KS = (5, 10, 20)
PRIMARY_K = 10
BOOTSTRAP_RESAMPLES = 2000
PAIR_SAMPLE_SIZE = 5000
MIN_SUBGROUP_SIZE = 30


## Step 2 — Load the frozen evaluation data and register authority


In [15]:
learners = pd.read_parquet(ASSIST_ROOT / 'learner_splits.parquet')
catalog = pd.read_parquet(ASSIST_ROOT / 'problem_catalog.parquet')
early_problem = pd.read_parquet(
    ASSIST_ROOT / 'early_problem_history.parquet'
)
early_skill = pd.read_parquet(ASSIST_ROOT / 'early_skill_history.parquet')
future_problem = pd.read_parquet(
    ASSIST_ROOT / 'future_problem_relevance.parquet'
)
problem_skill = pd.read_parquet(ASSIST_ROOT / 'problem_skill_map.parquet')
recommendations = pd.read_parquet(
    PHASE4_ROOT / 'hybrid_recommendations.parquet'
)
contributions = pd.read_parquet(
    PHASE4_ROOT / 'hybrid_component_contributions.parquet'
)

for frame in [
    learners, early_problem, early_skill, future_problem,
    recommendations, contributions,
]:
    if 'learner_id' in frame:
        frame['learner_id'] = frame['learner_id'].astype(str)
for frame in [catalog, early_problem, future_problem, recommendations, contributions]:
    if 'item_id' in frame:
        frame['item_id'] = frame['item_id'].astype(str)
problem_skill['problem_item_id'] = problem_skill['problem_item_id'].astype(str)
problem_skill['skill_item_id'] = problem_skill['skill_item_id'].astype(str)

phase3_dataset = pds.dataset(
    BASELINE_ROOT / 'validation_recommendations.parquet', format='parquet'
)
base_expression = (
    (pds.field('Dataset') == 'ASSISTments')
    & (pds.field('Task') == 'problem')
    & pds.field('Model').isin([
        PRIMARY_COMPARATOR, 'popularity_discovery_early'
    ])
)
comparator_recommendations = phase3_dataset.to_table(
    filter=base_expression,
    columns=[
        'Model', 'CandidatePolicy', 'RelevanceDefinition',
        'learner_id', 'item_id', 'score', 'rank', 'score_source',
    ],
).to_pandas()
comparator_recommendations['learner_id'] = (
    comparator_recommendations['learner_id'].astype(str)
)
comparator_recommendations['item_id'] = (
    comparator_recommendations['item_id'].astype(str)
)

validation = learners[learners['cohort'].eq('validation')].copy()
validation_ids = set(validation['learner_id'])

evaluation_contract = {
    'primary_model': PRODUCTION_MODEL,
    'primary_comparator': PRIMARY_COMPARATOR,
    'primary_relevance': 'attempted',
    'primary_policy': 'all_supported',
    'primary_k': PRIMARY_K,
    'sensitivity_relevance': 'successful',
    'sensitivity_policy': 'novel_only',
    'ks': list(KS),
}
display(pd.json_normalize(evaluation_contract).T.rename(columns={0: 'Value'}))

,Value
primary_model,gated_cf_content_with_sequential_fallback
primary_comparator,learner_neighbor_cf
primary_relevance,attempted
primary_policy,all_supported
primary_k,10
sensitivity_relevance,successful
sensitivity_policy,novel_only
ks,"[5, 10, 20]"


## Step 3 — Calculate core ranking metrics


In [4]:
RELEVANCE_COLUMNS = {
    'attempted': 'relevance_binary',
    'successful': 'successful_future_item',
}

def user_ranking_metrics(recommended, relevant, k):
    items = list(recommended[:k])
    relevant = set(relevant)
    hits = np.array([item in relevant for item in items], dtype=float)
    hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    discounts = np.log2(np.arange(2, k + 2))
    ideal_length = min(len(relevant), k)
    idcg = np.sum(1.0 / discounts[:ideal_length])
    positions = np.flatnonzero(hits)
    return {
        'PrecisionAtK': hits.sum() / k,
        'RecallAtK': hits.sum() / len(relevant),
        'NDCGAtK': np.sum(hits / discounts) / idcg if idcg else 0.0,
        'MAPAtK': (
            sum(hits[:pos + 1].sum() / (pos + 1) for pos in positions)
            / ideal_length
        ),
        'HitRateAtK': float(hits.sum() > 0),
    }

def relevant_sets(relevance_name, policy):
    column = RELEVANCE_COLUMNS[relevance_name]
    mask = (
        future_problem['in_candidate_catalog']
        & future_problem[column].eq(1)
    )
    if policy == 'novel_only':
        mask &= ~future_problem['seen_in_early'].fillna(False).astype(bool)
    relevant = future_problem.loc[
        mask, ['learner_id', 'item_id']
    ].drop_duplicates()
    return relevant.groupby('learner_id')['item_id'].agg(set).to_dict()

cold_lookup = validation.set_index('learner_id')[
    'cold_start_problem_history'
].to_dict()

def evaluate_model(frame, model_name):
    summaries = []
    per_user_rows = []
    for (policy, relevance_name), group in frame.groupby([
        'CandidatePolicy', 'RelevanceDefinition'
    ]):
        rel = relevant_sets(relevance_name, policy)
        eligible = (
            set(validation.loc[validation['evaluable_problem'], 'learner_id'])
            & set(rel)
        )
        ranked = (
            group.sort_values(['learner_id', 'rank', 'item_id'])
            .groupby('learner_id')['item_id'].agg(list).to_dict()
        )
        segment_users = {
            'all': eligible,
            'cold_start': eligible & set(validation.loc[
                validation['cold_start_problem_history'], 'learner_id'
            ]),
            'non_cold_start': eligible & set(validation.loc[
                ~validation['cold_start_problem_history'], 'learner_id'
            ]),
        }
        for k in KS:
            base_rows = {}
            for learner_id in sorted(eligible):
                values = user_ranking_metrics(
                    ranked.get(learner_id, []), rel[learner_id], k
                )
                base_rows[learner_id] = {
                    **values,
                    'RecommendedCount': len(ranked.get(learner_id, [])[:k]),
                    'RelevantCount': len(rel[learner_id]),
                }
                per_user_rows.append({
                    'Model': model_name, 'CandidatePolicy': policy,
                    'RelevanceDefinition': relevance_name, 'K': k,
                    'learner_id': learner_id,
                    'ColdStart': bool(cold_lookup[learner_id]),
                    **base_rows[learner_id],
                })
            for segment, users in segment_users.items():
                values = pd.DataFrame([
                    base_rows[user] for user in sorted(users)
                ])
                union = set()
                for user in users:
                    union.update(ranked.get(user, [])[:k])
                summaries.append({
                    'Model': model_name, 'CandidatePolicy': policy,
                    'RelevanceDefinition': relevance_name,
                    'Segment': segment, 'K': k,
                    **values[[
                        'PrecisionAtK', 'RecallAtK', 'NDCGAtK',
                        'MAPAtK', 'HitRateAtK',
                    ]].mean().to_dict(),
                    'CatalogCoverageAtK': len(union) / len(catalog),
                    'MeanRecommendations': values['RecommendedCount'].mean(),
                    'EvaluatedLearners': len(users),
                })
    return pd.DataFrame(summaries), pd.DataFrame(per_user_rows)

promoted_summary, promoted_users = evaluate_model(
    recommendations, PRODUCTION_MODEL
)
cf_rows = comparator_recommendations[
    comparator_recommendations['Model'].eq(PRIMARY_COMPARATOR)
]
cf_summary, cf_users = evaluate_model(cf_rows, PRIMARY_COMPARATOR)
ranking_metrics = pd.concat(
    [promoted_summary, cf_summary], ignore_index=True
)

ranking_metrics.to_csv(
    OUTPUT_ROOT / 'ranking_metrics.csv', index=False
)
display(ranking_metrics[
    ranking_metrics['Segment'].eq('all')
    & ranking_metrics['K'].eq(10)
])

,Model,CandidatePolicy,RelevanceDefinition,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners
3,gated_cf_content_with_sequential_fallback,all_supported,attempted,all,10,0.439710,0.270500,0.498745,0.454452,0.671448,0.224088,9.883089,5996
12,gated_cf_content_with_sequential_fallback,all_supported,successful,all,10,0.333225,0.275282,0.413926,0.344002,0.648579,0.221800,9.883088,5842
21,gated_cf_content_with_sequential_fallback,novel_only,attempted,all,10,0.445987,0.285510,0.510959,0.470238,0.679315,0.242490,9.793989,5956
30,gated_cf_content_with_sequential_fallback,novel_only,successful,all,10,0.337274,0.291240,0.425440,0.358521,0.651251,0.237613,9.800345,5795
39,learner_neighbor_cf,all_supported,attempted,all,10,0.439626,0.270136,0.498392,0.454147,0.670781,0.219513,9.796197,5996
48,learner_neighbor_cf,all_supported,successful,all,10,0.333122,0.274242,0.412909,0.343005,0.647381,0.217074,9.810852,5842
57,learner_neighbor_cf,novel_only,attempted,all,10,0.445937,0.285150,0.510575,0.469871,0.678811,0.236632,9.678643,5956
66,learner_neighbor_cf,novel_only,successful,all,10,0.337153,0.290176,0.424387,0.357493,0.650043,0.231529,9.705608,5795


## Step 4 — Evaluate catalogue reach and list completeness


In [5]:
def list_health_for_slice(group, policy, relevance_name, k):
    rel = relevant_sets(relevance_name, policy)
    eligible = (
        set(validation.loc[validation['evaluable_problem'], 'learner_id'])
        & set(rel)
    )
    lengths = group[group['rank'].le(k)].groupby('learner_id').size()
    seen_counts = early_problem[
        early_problem['learner_id'].isin(eligible)
        & early_problem['in_candidate_catalog']
    ].groupby('learner_id')['item_id'].nunique()
    rows = []
    for learner_id in sorted(eligible):
        count = int(lengths.get(learner_id, 0))
        remaining = (
            len(catalog)
            if policy == 'all_supported'
            else len(catalog) - int(seen_counts.get(learner_id, 0))
        )
        if count >= k:
            reason = 'complete'
        elif remaining < k:
            reason = 'catalogue_exhaustion'
        else:
            reason = 'insufficient_model_candidates'
        rows.append({
            'CandidatePolicy': policy,
            'RelevanceDefinition': relevance_name,
            'K': k, 'learner_id': learner_id,
            'RecommendationCount': count,
            'ShortListReason': reason,
        })
    return pd.DataFrame(rows)

list_health_parts = []
for (policy, relevance_name), group in recommendations.groupby([
    'CandidatePolicy', 'RelevanceDefinition'
]):
    for k in KS:
        list_health_parts.append(
            list_health_for_slice(group, policy, relevance_name, k)
        )
list_health_users = pd.concat(list_health_parts, ignore_index=True)
list_health_users = list_health_users.merge(
    validation[['learner_id', 'cold_start_problem_history']],
    on='learner_id', how='left', validate='many_to_one',
)

coverage_rows = []
for (policy, relevance_name, k), group in list_health_users.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'K'
]):
    for segment, mask in {
        'all': pd.Series(True, index=group.index),
        'cold_start': group['cold_start_problem_history'],
        'non_cold_start': ~group['cold_start_problem_history'],
    }.items():
        subset = group[mask]
        rec_slice = recommendations[
            recommendations['CandidatePolicy'].eq(policy)
            & recommendations['RelevanceDefinition'].eq(relevance_name)
            & recommendations['rank'].le(k)
            & recommendations['learner_id'].isin(subset['learner_id'])
        ]
        coverage_rows.append({
            'CandidatePolicy': policy,
            'RelevanceDefinition': relevance_name,
            'K': k, 'Segment': segment,
            'Learners': len(subset),
            'UniqueRecommendedItems': rec_slice['item_id'].nunique(),
            'CatalogCoverage': rec_slice['item_id'].nunique() / len(catalog),
            'MeanListLength': subset['RecommendationCount'].mean(),
            'MedianListLength': subset['RecommendationCount'].median(),
            'UnderKLearnerRate': subset['RecommendationCount'].lt(k).mean(),
            'NoRecommendationRate': subset['RecommendationCount'].eq(0).mean(),
            'CatalogueExhaustionLearners': subset[
                'ShortListReason'
            ].eq('catalogue_exhaustion').sum(),
            'InsufficientCandidateLearners': subset[
                'ShortListReason'
            ].eq('insufficient_model_candidates').sum(),
        })
coverage_and_list_health = pd.DataFrame(coverage_rows)
coverage_and_list_health.to_csv(
    OUTPUT_ROOT / 'coverage_and_list_health.csv', index=False
)
display(coverage_and_list_health[
    coverage_and_list_health['K'].eq(10)
])

,CandidatePolicy,RelevanceDefinition,K,Segment,Learners,UniqueRecommendedItems,CatalogCoverage,MeanListLength,MedianListLength,UnderKLearnerRate,NoRecommendationRate,CatalogueExhaustionLearners,InsufficientCandidateLearners
3,all_supported,attempted,10,all,5996,8914,0.224088,9.883089,10.0,0.022181,0.0,0,133
4,all_supported,attempted,10,cold_start,21,84,0.002112,10.000000,10.0,0.000000,0.0,0,0
5,all_supported,attempted,10,non_cold_start,5975,8854,0.222580,9.882678,10.0,0.022259,0.0,0,133
12,all_supported,successful,10,all,5842,8823,0.221800,9.883088,10.0,0.024307,0.0,0,142
13,all_supported,successful,10,cold_start,15,68,0.001709,10.000000,10.0,0.000000,0.0,0,0
14,all_supported,successful,10,non_cold_start,5827,8776,0.220619,9.882787,10.0,0.024369,0.0,0,142
21,novel_only,attempted,10,all,5956,9646,0.242490,9.793989,10.0,0.035930,0.0,0,214
22,novel_only,attempted,10,cold_start,21,84,0.002112,10.000000,10.0,0.000000,0.0,0,0
23,novel_only,attempted,10,non_cold_start,5935,9591,0.241107,9.793260,10.0,0.036057,0.0,0,214
30,novel_only,successful,10,all,5795,9452,0.237613,9.800345,10.0,0.035203,0.0,0,204


## Step 5 — Add association-weighted recommendation diversity


In [6]:
skill_weights = {}
for item_id, group in problem_skill.groupby('problem_item_id'):
    skill_weights[item_id] = dict(zip(
        group['skill_item_id'], group['association_share'].astype(float)
    ))

catalog_metadata = catalog.set_index('item_id')
metadata_tokens = {}
for item_id, row in catalog_metadata.iterrows():
    tokens = set()
    for column in ['problem_type', 'hierarchy']:
        if column in row.index and pd.notna(row[column]):
            tokens.add(f'{column}:{row[column]}')
    metadata_tokens[item_id] = tokens

def weighted_jaccard(item_a, item_b):
    left = skill_weights.get(item_a, {})
    right = skill_weights.get(item_b, {})
    if left and right:
        keys = set(left) | set(right)
        numerator = sum(min(left.get(key, 0), right.get(key, 0)) for key in keys)
        denominator = sum(max(left.get(key, 0), right.get(key, 0)) for key in keys)
        return numerator / denominator if denominator else 0.0
    left_tokens = metadata_tokens.get(item_a, set())
    right_tokens = metadata_tokens.get(item_b, set())
    union = left_tokens | right_tokens
    return len(left_tokens & right_tokens) / len(union) if union else 0.0

def list_diversity(items):
    pairs = list(combinations(items, 2))
    if not pairs:
        return np.nan
    return 1.0 - np.mean([
        weighted_jaccard(left, right) for left, right in pairs
    ])

diversity_user_rows = []
for (policy, relevance_name), group in recommendations.groupby([
    'CandidatePolicy', 'RelevanceDefinition'
]):
    ranked = (
        group.sort_values(['learner_id', 'rank'])
        .groupby('learner_id')['item_id'].agg(list).to_dict()
    )
    rel = relevant_sets(relevance_name, policy)
    eligible = (
        set(validation.loc[validation['evaluable_problem'], 'learner_id'])
        & set(rel)
    )
    for k in KS:
        for learner_id in sorted(eligible):
            items = ranked.get(learner_id, [])[:k]
            represented_skills = set()
            for item in items:
                represented_skills.update(skill_weights.get(item, {}))
            diversity_user_rows.append({
                'CandidatePolicy': policy,
                'RelevanceDefinition': relevance_name, 'K': k,
                'learner_id': learner_id,
                'IntraListDiversity': list_diversity(items),
                'RepresentedSkills': len(represented_skills),
            })
diversity_users = pd.DataFrame(diversity_user_rows).merge(
    validation[['learner_id', 'cold_start_problem_history']],
    on='learner_id', how='left', validate='many_to_one',
)

diversity_rows = []
supported_skills = set(problem_skill['skill_item_id'])
for keys, group in diversity_users.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'K'
]):
    policy, relevance_name, k = keys
    for segment, mask in {
        'all': pd.Series(True, index=group.index),
        'cold_start': group['cold_start_problem_history'],
        'non_cold_start': ~group['cold_start_problem_history'],
    }.items():
        subset = group[mask]
        diversity_rows.append({
            'CandidatePolicy': policy,
            'RelevanceDefinition': relevance_name, 'K': k,
            'Segment': segment, 'Learners': len(subset),
            'DefinedDiversityLearners': subset[
                'IntraListDiversity'
            ].notna().sum(),
            'MeanIntraListDiversity': subset[
                'IntraListDiversity'
            ].mean(),
            'MedianIntraListDiversity': subset[
                'IntraListDiversity'
            ].median(),
            'DiversityQ25': subset['IntraListDiversity'].quantile(0.25),
            'DiversityQ75': subset['IntraListDiversity'].quantile(0.75),
            'ZeroDiversityRate': subset[
                'IntraListDiversity'
            ].eq(0).mean(),
            'MeanRepresentedSkills': subset['RepresentedSkills'].mean(),
        })
diversity_metrics = pd.DataFrame(diversity_rows)
diversity_metrics.to_csv(
    OUTPUT_ROOT / 'diversity_metrics.csv', index=False
)
display(diversity_metrics[diversity_metrics['K'].eq(10)])

,CandidatePolicy,RelevanceDefinition,K,Segment,Learners,DefinedDiversityLearners,MeanIntraListDiversity,MedianIntraListDiversity,DiversityQ25,DiversityQ75,ZeroDiversityRate,MeanRepresentedSkills
3,all_supported,attempted,10,all,5996,5974,0.543453,0.592593,0.355556,0.777778,0.106404,2.304703
4,all_supported,attempted,10,cold_start,21,21,0.255026,0.355556,0.000000,0.355556,0.333333,0.523810
5,all_supported,attempted,10,non_cold_start,5975,5953,0.544470,0.592593,0.355556,0.777778,0.105607,2.310962
12,all_supported,successful,10,all,5842,5824,0.580815,0.644444,0.400000,0.800000,0.070866,2.402431
13,all_supported,successful,10,cold_start,15,15,0.262222,0.355556,0.000000,0.355556,0.333333,0.600000
14,all_supported,successful,10,non_cold_start,5827,5809,0.581637,0.644444,0.400000,0.800000,0.070190,2.407071
21,novel_only,attempted,10,all,5956,5915,0.550256,0.600000,0.355556,0.777778,0.101914,2.356783
22,novel_only,attempted,10,cold_start,21,21,0.255026,0.355556,0.000000,0.355556,0.333333,0.523810
23,novel_only,attempted,10,non_cold_start,5935,5894,0.551308,0.600000,0.355556,0.777778,0.101095,2.363269
30,novel_only,successful,10,all,5795,5754,0.587977,0.644444,0.414815,0.800000,0.065401,2.474547


## Step 6 — Add novelty and repetition diagnostics


In [7]:
catalog_novelty = catalog[[
    'item_id', 'training_interactions'
]].copy()
total_training = catalog_novelty['training_interactions'].sum()
catalog_novelty['DiscoveryPopularityProbability'] = (
    catalog_novelty['training_interactions'] / total_training
)
minimum_probability = 1.0 / (total_training + len(catalog_novelty))
catalog_novelty['NoveltyBits'] = -np.log2(
    catalog_novelty['DiscoveryPopularityProbability'].clip(
        lower=minimum_probability
    )
)
catalog_novelty['PopularityDecile'] = pd.qcut(
    catalog_novelty['training_interactions'].rank(method='first'),
    10, labels=[f'D{i}' for i in range(1, 11)]
)

seen_pairs = early_problem[
    early_problem['learner_id'].isin(validation_ids)
    & early_problem['in_candidate_catalog']
][['learner_id', 'item_id']].drop_duplicates().assign(SeenEarly=True)
novelty_rows = recommendations.merge(
    catalog_novelty, on='item_id', how='left', validate='many_to_one'
).merge(
    seen_pairs, on=['learner_id', 'item_id'], how='left',
    validate='many_to_one',
)
novelty_rows['SeenEarly'] = novelty_rows['SeenEarly'].fillna(False)

novelty_user_parts = []
for (policy, relevance_name), group in novelty_rows.groupby([
    'CandidatePolicy', 'RelevanceDefinition'
]):
    for k in KS:
        top = group[group['rank'].le(k)]
        observed = top.groupby('learner_id').agg(
            MeanNoveltyBits=('NoveltyBits', 'mean'),
            SeenItemRate=('SeenEarly', 'mean'),
            RecommendationCount=('item_id', 'size'),
        ).reset_index()
        rel = relevant_sets(relevance_name, policy)
        eligible = sorted(
            set(validation.loc[
                validation['evaluable_problem'], 'learner_id'
            ]) & set(rel)
        )
        users = pd.DataFrame({'learner_id': eligible}).merge(
            observed, on='learner_id', how='left', validate='one_to_one'
        )
        users['RecommendationCount'] = (
            users['RecommendationCount'].fillna(0).astype(int)
        )
        users['SeenItemRate'] = users['SeenItemRate'].fillna(0.0)
        users['CandidatePolicy'] = policy
        users['RelevanceDefinition'] = relevance_name
        users['K'] = k
        novelty_user_parts.append(users)
novelty_users = pd.concat(novelty_user_parts, ignore_index=True).merge(
    validation[['learner_id', 'cold_start_problem_history']],
    on='learner_id', how='left', validate='many_to_one',
)
novelty_summary_rows = []
for keys, group in novelty_users.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'K'
]):
    policy, relevance_name, k = keys
    for segment, mask in {
        'all': pd.Series(True, index=group.index),
        'cold_start': group['cold_start_problem_history'],
        'non_cold_start': ~group['cold_start_problem_history'],
    }.items():
        subset = group[mask]
        novelty_summary_rows.append({
            'CandidatePolicy': policy,
            'RelevanceDefinition': relevance_name, 'K': k,
            'Segment': segment, 'Learners': len(subset),
            'MeanNoveltyBits': subset['MeanNoveltyBits'].mean(),
            'MedianNoveltyBits': subset['MeanNoveltyBits'].median(),
            'MeanSeenItemRate': subset['SeenItemRate'].fillna(0).mean(),
            'NoRecommendationRate': subset[
                'RecommendationCount'
            ].fillna(0).eq(0).mean(),
        })
novelty_metrics = pd.DataFrame(novelty_summary_rows)
novelty_metrics.to_csv(OUTPUT_ROOT / 'novelty_metrics.csv', index=False)

popularity_distribution = novelty_rows.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'PopularityDecile'
], observed=True).size().rename('RecommendationRows').reset_index()
popularity_distribution.to_csv(
    OUTPUT_ROOT / 'recommended_popularity_deciles.csv', index=False
)
display(novelty_metrics[novelty_metrics['K'].eq(10)])

/tmp/ipykernel_1031/878108311.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  novelty_rows['SeenEarly'] = novelty_rows['SeenEarly'].fillna(False)


,CandidatePolicy,RelevanceDefinition,K,Segment,Learners,MeanNoveltyBits,MedianNoveltyBits,MeanSeenItemRate,NoRecommendationRate
3,all_supported,attempted,10,all,5996,14.609685,14.909204,0.124243,0.0
4,all_supported,attempted,10,cold_start,21,12.621142,10.041960,0.000000,0.0
5,all_supported,attempted,10,non_cold_start,5975,14.616674,14.909215,0.124680,0.0
12,all_supported,successful,10,all,5842,14.597366,14.919398,0.122437,0.0
13,all_supported,successful,10,cold_start,15,12.837279,10.041960,0.000000,0.0
14,all_supported,successful,10,non_cold_start,5827,14.601897,14.919491,0.122752,0.0
21,novel_only,attempted,10,all,5956,14.623913,14.926717,0.000000,0.0
22,novel_only,attempted,10,cold_start,21,12.621142,10.041960,0.000000,0.0
23,novel_only,attempted,10,non_cold_start,5935,14.630999,14.927929,0.000000,0.0
30,novel_only,successful,10,all,5795,14.612846,14.926428,0.000000,0.0


## Step 7 — Measure personalization with sampled learner pairs


In [8]:
def sampled_personalization(frame, model_name):
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    for (policy, relevance_name), group in frame.groupby([
        'CandidatePolicy', 'RelevanceDefinition'
    ]):
        cold_lookup = validation.set_index('learner_id')[
            'cold_start_problem_history'
        ].to_dict()
        for k in KS:
            lists = (
                group[group['rank'].le(k)]
                .sort_values(['learner_id', 'rank'])
                .groupby('learner_id')['item_id']
                .agg(lambda values: tuple(values)).to_dict()
            )
            for segment in ['all', 'cold_start', 'non_cold_start']:
                users = sorted(lists)
                if segment == 'cold_start':
                    users = [u for u in users if cold_lookup.get(u, False)]
                elif segment == 'non_cold_start':
                    users = [u for u in users if not cold_lookup.get(u, False)]
                if len(users) < 2:
                    continue
                left = rng.integers(0, len(users), PAIR_SAMPLE_SIZE)
                right = rng.integers(0, len(users), PAIR_SAMPLE_SIZE)
                valid = left != right
                overlaps = []
                for i, j in zip(left[valid], right[valid]):
                    a, b = set(lists[users[i]]), set(lists[users[j]])
                    overlaps.append(len(a & b) / len(a | b) if a | b else 0.0)
                list_series = pd.Series([lists[user] for user in users])
                rows.append({
                    'Model': model_name, 'CandidatePolicy': policy,
                    'RelevanceDefinition': relevance_name,
                    'K': k, 'Segment': segment, 'Learners': len(users),
                    'SampledPairs': len(overlaps),
                    'MeanJaccardOverlap': np.mean(overlaps),
                    'Personalization': 1.0 - np.mean(overlaps),
                    'UniqueListRate': list_series.nunique() / len(list_series),
                    'MostCommonListLearners': list_series.value_counts().iloc[0],
                })
    return pd.DataFrame(rows)

promoted_personalization = sampled_personalization(
    recommendations, PRODUCTION_MODEL
)
popularity_personalization = sampled_personalization(
    comparator_recommendations[
        comparator_recommendations['Model'].eq(
            'popularity_discovery_early'
        )
    ],
    'popularity_discovery_early',
)
personalization_metrics = pd.concat([
    promoted_personalization, popularity_personalization
], ignore_index=True)
personalization_metrics.to_csv(
    OUTPUT_ROOT / 'personalization_metrics.csv', index=False
)
display(personalization_metrics[
    personalization_metrics['K'].eq(10)
    & personalization_metrics['Segment'].eq('all')
])

,Model,CandidatePolicy,RelevanceDefinition,K,Segment,Learners,SampledPairs,MeanJaccardOverlap,Personalization,UniqueListRate,MostCommonListLearners
3,gated_cf_content_with_sequential_fallback,all_supported,attempted,10,all,6667,5000,0.002103,0.997897,0.788061,141
12,gated_cf_content_with_sequential_fallback,all_supported,successful,10,all,6667,5000,0.004088,0.995912,0.868607,141
21,gated_cf_content_with_sequential_fallback,novel_only,attempted,10,all,6667,4999,0.004575,0.995425,0.809210,310
30,gated_cf_content_with_sequential_fallback,novel_only,successful,10,all,6667,4998,0.004198,0.995802,0.873256,316
39,popularity_discovery_early,all_supported,attempted,10,all,6667,5000,1.000000,0.000000,0.000150,6667
48,popularity_discovery_early,all_supported,successful,10,all,6667,5000,1.000000,0.000000,0.000150,6667
57,popularity_discovery_early,novel_only,attempted,10,all,6667,4999,0.815619,0.184381,0.090445,5917
66,popularity_discovery_early,novel_only,successful,10,all,6667,4998,0.808279,0.191721,0.090445,5917


## Step 8 — Audit routes and component behavior


In [9]:
route_rows = []
for keys, group in contributions[
    contributions['rank'].le(10)
].groupby(['CandidatePolicy', 'RelevanceDefinition', 'learner_id']):
    policy, relevance_name, learner_id = keys
    cf_used = group['cf_contribution'].gt(0).any()
    sequence_used = group['sequential_contribution'].gt(0).any()
    content_used = group['content_contribution'].gt(0).any()
    fallback_used = group['FallbackUsed'].any()
    if fallback_used:
        route = 'popularity_fallback'
    elif sequence_used:
        route = 'sequential_route'
    elif cf_used:
        route = 'cf_route'
    elif content_used:
        route = 'content_only'
    else:
        route = 'no_personalized_evidence'
    route_rows.append({
        'CandidatePolicy': policy,
        'RelevanceDefinition': relevance_name,
        'learner_id': learner_id, 'Route': route,
        'CFUsed': cf_used, 'SequenceUsed': sequence_used,
        'ContentUsed': content_used, 'FallbackUsed': fallback_used,
        'RecommendationCount': len(group),
    })
route_users = pd.DataFrame(route_rows)
extended_users = (
    promoted_users[promoted_users['K'].eq(10)]
    .merge(
        route_users,
        on=['CandidatePolicy', 'RelevanceDefinition', 'learner_id'],
        how='left', validate='one_to_one',
    )
    .merge(
        diversity_users[diversity_users['K'].eq(10)][[
            'CandidatePolicy', 'RelevanceDefinition', 'learner_id',
            'IntraListDiversity',
        ]],
        on=['CandidatePolicy', 'RelevanceDefinition', 'learner_id'],
        how='left', validate='one_to_one',
    )
    .merge(
        novelty_users[novelty_users['K'].eq(10)][[
            'CandidatePolicy', 'RelevanceDefinition', 'learner_id',
            'MeanNoveltyBits',
        ]],
        on=['CandidatePolicy', 'RelevanceDefinition', 'learner_id'],
        how='left', validate='one_to_one',
    )
)
extended_users['Route'] = extended_users['Route'].fillna('no_recommendation')
for column in ['CFUsed', 'SequenceUsed', 'ContentUsed', 'FallbackUsed']:
    extended_users[column] = extended_users[column].fillna(False)
route_metrics = extended_users.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'Route'
]).agg(
    Learners=('learner_id', 'size'),
    RecallAt10=('RecallAtK', 'mean'),
    NDCGAt10=('NDCGAtK', 'mean'),
    HitRateAt10=('HitRateAtK', 'mean'),
    MeanListLength=('RecommendedCount', 'mean'),
    MeanDiversity=('IntraListDiversity', 'mean'),
    MeanNoveltyBits=('MeanNoveltyBits', 'mean'),
    ContentUserRate=('ContentUsed', 'mean'),
).reset_index()
route_coverage_rows = []
for keys, group in extended_users.groupby([
    'CandidatePolicy', 'RelevanceDefinition', 'Route'
]):
    policy, relevance_name, route = keys
    rec_slice = recommendations[
        recommendations['CandidatePolicy'].eq(policy)
        & recommendations['RelevanceDefinition'].eq(relevance_name)
        & recommendations['rank'].le(10)
        & recommendations['learner_id'].isin(group['learner_id'])
    ]
    route_coverage_rows.append({
        'CandidatePolicy': policy,
        'RelevanceDefinition': relevance_name, 'Route': route,
        'UniqueRecommendedItems': rec_slice['item_id'].nunique(),
        'CatalogCoverageAt10': rec_slice['item_id'].nunique() / len(catalog),
    })
route_metrics = route_metrics.merge(
    pd.DataFrame(route_coverage_rows),
    on=['CandidatePolicy', 'RelevanceDefinition', 'Route'],
    how='left', validate='one_to_one',
)
route_metrics['ComparisonAuthority'] = 'descriptive_nonrandom_routes'
route_metrics.to_csv(OUTPUT_ROOT / 'route_metrics.csv', index=False)
display(route_metrics[
    route_metrics['CandidatePolicy'].eq('all_supported')
    & route_metrics['RelevanceDefinition'].eq('attempted')
])

,CandidatePolicy,RelevanceDefinition,Route,Learners,RecallAt10,NDCGAt10,HitRateAt10,MeanListLength,MeanDiversity,MeanNoveltyBits,ContentUserRate,UniqueRecommendedItems,CatalogCoverageAt10,ComparisonAuthority
0,all_supported,attempted,cf_route,5971,0.271260,0.500442,0.673756,9.886451,0.544666,14.615692,0.105175,8847,0.222404,descriptive_nonrandom_routes
1,all_supported,attempted,content_only,9,0.111111,0.111111,0.111111,10.000000,0.120988,16.060053,1.000000,74,0.001860,descriptive_nonrandom_routes
2,all_supported,attempted,popularity_fallback,12,0.000000,0.000000,0.000000,10.000000,0.355556,10.041960,0.000000,10,0.000251,descriptive_nonrandom_routes
3,all_supported,attempted,sequential_route,4,0.305556,0.334069,0.500000,4.250000,0.155556,16.082554,0.250000,17,0.000427,descriptive_nonrandom_routes


## Step 9 — Evaluate early-evidence subgroup robustness


In [10]:
primary_users = extended_users[
    extended_users['CandidatePolicy'].eq('all_supported')
    & extended_users['RelevanceDefinition'].eq('attempted')
].copy()

history_counts = early_problem[
    early_problem['learner_id'].isin(validation_ids)
    & early_problem['in_candidate_catalog']
].groupby('learner_id')['early_interaction_count'].sum()
skill_evidence = early_skill[
    early_skill['learner_id'].isin(validation_ids)
    & early_skill['in_candidate_catalog']
    & early_skill['mastery_evidence_confidence'].gt(0)
].groupby('learner_id').size()
primary_users['EarlyInteractions'] = (
    primary_users['learner_id'].map(history_counts).fillna(0)
)
primary_users['HasSupportedSkillEvidence'] = (
    primary_users['learner_id'].map(skill_evidence).fillna(0).gt(0)
)
primary_users['ColdStartGroup'] = np.where(
    primary_users['ColdStart'], 'cold_start', 'non_cold_start'
)
primary_users['HistoryBand'] = 'cold_start'
non_cold = primary_users['EarlyInteractions'].gt(0)
primary_users.loc[non_cold, 'HistoryBand'] = pd.qcut(
    primary_users.loc[non_cold, 'EarlyInteractions'].rank(method='first'),
    4, labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'],
).astype(str)
primary_users['RelevantSetBand'] = pd.qcut(
    primary_users['RelevantCount'].rank(method='first'),
    4, labels=['Q1_small', 'Q2', 'Q3', 'Q4_large'],
).astype(str)

subgroup_rows = []
dimensions = {
    'ColdStart': 'ColdStartGroup',
    'EarlyHistory': 'HistoryBand',
    'SkillEvidence': 'HasSupportedSkillEvidence',
    'RelevantSetSize': 'RelevantSetBand',
}
for dimension, column in dimensions.items():
    for value, group in primary_users.groupby(column, dropna=False):
        rec_slice = recommendations[
            recommendations['CandidatePolicy'].eq('all_supported')
            & recommendations['RelevanceDefinition'].eq('attempted')
            & recommendations['rank'].le(10)
            & recommendations['learner_id'].isin(group['learner_id'])
        ]
        subgroup_rows.append({
            'Dimension': dimension, 'Group': str(value),
            'Learners': len(group),
            'StableSummary': len(group) >= MIN_SUBGROUP_SIZE,
            'RecallAt10': group['RecallAtK'].mean(),
            'NDCGAt10': group['NDCGAtK'].mean(),
            'HitRateAt10': group['HitRateAtK'].mean(),
            'MeanListLength': group['RecommendedCount'].mean(),
            'MeanDiversity': group['IntraListDiversity'].mean(),
            'MeanNoveltyBits': group['MeanNoveltyBits'].mean(),
            'UniqueRecommendedItems': rec_slice['item_id'].nunique(),
            'CatalogCoverageAt10': (
                rec_slice['item_id'].nunique() / len(catalog)
            ),
            'FallbackRate': group['Route'].eq(
                'popularity_fallback'
            ).mean(),
        })
subgroup_metrics = pd.DataFrame(subgroup_rows)
subgroup_metrics.to_csv(OUTPUT_ROOT / 'subgroup_metrics.csv', index=False)
display(subgroup_metrics)

,Dimension,Group,Learners,StableSummary,RecallAt10,NDCGAt10,HitRateAt10,MeanListLength,MeanDiversity,MeanNoveltyBits,UniqueRecommendedItems,CatalogCoverageAt10,FallbackRate
0,ColdStart,cold_start,21,False,0.047619,0.047619,0.047619,10.000000,0.255026,12.621142,84,0.002112,0.571429
1,ColdStart,non_cold_start,5975,True,0.271283,0.500330,0.673640,9.882678,0.544470,14.616674,8854,0.222580,0.000000
2,EarlyHistory,Q1_low,1494,True,0.426517,0.438986,0.589023,9.735609,0.505671,14.683805,4502,0.113175,0.000000
3,EarlyHistory,Q2,1494,True,0.306429,0.432331,0.607764,9.940428,0.539294,14.659369,4508,0.113326,0.000000
4,EarlyHistory,Q3,1493,True,0.233655,0.513543,0.691226,9.854655,0.557670,14.426107,3888,0.097740,0.000000
5,EarlyHistory,Q4_high,1494,True,0.118507,0.616471,0.806560,10.000000,0.575066,14.697289,2906,0.073054,0.000000
6,EarlyHistory,cold_start,21,False,0.047619,0.047619,0.047619,10.000000,0.255026,12.621142,84,0.002112,0.571429
7,SkillEvidence,False,2043,True,0.450618,0.761737,0.872736,9.693098,0.393896,13.794140,2223,0.055884,0.003426
8,SkillEvidence,True,3953,True,0.177411,0.362824,0.567417,9.981280,0.619915,15.031178,7385,0.185651,0.001265
9,RelevantSetSize,Q1_small,1499,True,0.433240,0.414359,0.567045,9.540360,0.531405,15.053515,4975,0.125066,0.008005


## Step 10 — Quantify uncertainty and practical effect size


In [18]:
paired = promoted_users[
    promoted_users['CandidatePolicy'].eq('all_supported')
    & promoted_users['RelevanceDefinition'].eq('attempted')
    & promoted_users['K'].eq(10)
][['learner_id', 'RecallAtK', 'NDCGAtK']].merge(
    cf_users[
        cf_users['CandidatePolicy'].eq('all_supported')
        & cf_users['RelevanceDefinition'].eq('attempted')
        & cf_users['K'].eq(10)
    ][['learner_id', 'RecallAtK', 'NDCGAtK']],
    on='learner_id', suffixes=('_Promoted', '_CF'), validate='one_to_one',
)
paired['RecallDifference'] = (
    paired['RecallAtK_Promoted'] - paired['RecallAtK_CF']
)
paired['NDCGDifference'] = (
    paired['NDCGAtK_Promoted'] - paired['NDCGAtK_CF']
)

def bootstrap_summary(values, metric, comparison):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(RANDOM_STATE)
    means = np.empty(BOOTSTRAP_RESAMPLES)
    for index in range(BOOTSTRAP_RESAMPLES):
        sample = rng.integers(0, len(values), len(values))
        means[index] = values[sample].mean()
    lower, upper = np.quantile(means, [0.025, 0.975])
    return {
        'Metric': metric, 'Comparison': comparison,
        'Learners': len(values), 'Mean': values.mean(),
        'Median': np.median(values),
        'Lower95': lower, 'Upper95': upper,
        'WinRate': np.mean(values > 0),
        'TieRate': np.mean(values == 0),
        'LossRate': np.mean(values < 0),
        'Resamples': BOOTSTRAP_RESAMPLES, 'Seed': RANDOM_STATE,
    }

uncertainty_rows = [
    bootstrap_summary(
        paired['RecallDifference'], 'RecallAt10',
        'promoted_minus_neighbor_cf',
    ),
    bootstrap_summary(
        paired['NDCGDifference'], 'NDCGAt10',
        'promoted_minus_neighbor_cf',
    ),
]
primary_extended = primary_users.dropna(
    subset=['IntraListDiversity', 'MeanNoveltyBits']
)
uncertainty_rows.extend([
    bootstrap_summary(
        primary_extended['IntraListDiversity'],
        'IntraListDiversityAt10', 'promoted_level',
    ),
    bootstrap_summary(
        primary_extended['MeanNoveltyBits'],
        'NoveltyBitsAt10', 'promoted_level',
    ),
])
for route, group in primary_extended.groupby('Route'):
    if len(group) < MIN_SUBGROUP_SIZE:
        continue
    for column, metric in [
        ('NDCGAtK', 'NDCGAt10'),
        ('IntraListDiversity', 'IntraListDiversityAt10'),
        ('MeanNoveltyBits', 'NoveltyBitsAt10'),
    ]:
        uncertainty_rows.append(bootstrap_summary(
            group[column].dropna(), metric, f'route_level:{route}'
        ))
paired_uncertainty = pd.DataFrame(uncertainty_rows)
paired_uncertainty.to_csv(
    OUTPUT_ROOT / 'paired_uncertainty.csv', index=False
)
display(paired_uncertainty)

,Metric,Comparison,Learners,Mean,Median,Lower95,Upper95,WinRate,TieRate,LossRate,Resamples,Seed
0,RecallAt10,promoted_minus_neighbor_cf,5996,0.000363,0.000000,-0.000008,0.000871,0.000834,0.998999,0.000167,2000,42
1,NDCGAt10,promoted_minus_neighbor_cf,5996,0.000353,0.000000,-0.000048,0.000868,0.003502,0.993662,0.002835,2000,42
2,IntraListDiversityAt10,promoted_level,5974,0.543453,0.592593,0.535878,0.550297,0.893204,0.106796,0.000000,2000,42
3,NoveltyBitsAt10,promoted_level,5974,14.601179,14.903948,14.559637,14.646008,1.000000,0.000000,0.000000,2000,42
4,NDCGAt10,route_level:cf_route,5950,0.498679,0.502517,0.487266,0.510047,0.672605,0.327395,0.000000,2000,42
5,IntraListDiversityAt10,route_level:cf_route,5950,0.544666,0.592593,0.537053,0.551819,0.894286,0.105714,0.000000,2000,42
6,NoveltyBitsAt10,route_level:cf_route,5950,14.607593,14.904027,14.566289,14.650929,1.000000,0.000000,0.000000,2000,42


## Step 11 — Produce deterministic failure-case audit tables


In [12]:
ranked_primary = (
    recommendations[
        recommendations['CandidatePolicy'].eq('all_supported')
        & recommendations['RelevanceDefinition'].eq('attempted')
        & recommendations['rank'].le(10)
    ]
    .sort_values(['learner_id', 'rank'])
    .groupby('learner_id')['item_id'].agg(list).to_dict()
)
relevant_primary = relevant_sets('attempted', 'all_supported')
primary_contributions = contributions[
    contributions['CandidatePolicy'].eq('all_supported')
    & contributions['RelevanceDefinition'].eq('attempted')
    & contributions['rank'].le(10)
].groupby('learner_id').agg(
    CFContribution=('cf_contribution', 'sum'),
    SequentialContribution=('sequential_contribution', 'sum'),
    ContentContribution=('content_contribution', 'sum'),
).reset_index()
failure_base = primary_users.merge(
    paired[['learner_id', 'RecallDifference', 'NDCGDifference']],
    on='learner_id', how='left', validate='one_to_one',
).merge(
    primary_contributions, on='learner_id',
    how='left', validate='one_to_one',
)
failure_base['RecommendedItems'] = failure_base['learner_id'].map(
    ranked_primary
).apply(lambda value: value if isinstance(value, list) else [])
failure_base['HitPositions'] = failure_base.apply(
    lambda row: [
        index + 1 for index, item in enumerate(row['RecommendedItems'])
        if item in relevant_primary[row['learner_id']]
    ],
    axis=1,
)

cases = []
def add_cases(frame, label, count=10, ascending=True, column=None):
    selected = (
        frame.sort_values(
            [column, 'learner_id'] if column else ['learner_id'],
            ascending=[ascending, True] if column else True,
            kind='mergesort',
        ).head(count)
    )
    part = selected.copy()
    part['AuditCategory'] = label
    cases.append(part)

add_cases(
    failure_base[failure_base['HitRateAtK'].eq(0)],
    'no_hit', column='NDCGAtK',
)
add_cases(
    failure_base[failure_base['RecommendedCount'].lt(10)],
    'short_list', column='RecommendedCount',
)
add_cases(
    failure_base[failure_base['Route'].eq('popularity_fallback')],
    'popularity_fallback',
)
add_cases(
    failure_base[failure_base['Route'].eq('sequential_route')],
    'sequential_route',
)
add_cases(
    failure_base.dropna(subset=['IntraListDiversity']),
    'lowest_diversity', column='IntraListDiversity',
)
add_cases(
    failure_base.dropna(subset=['IntraListDiversity']),
    'highest_diversity', column='IntraListDiversity', ascending=False,
)
add_cases(
    failure_base, 'largest_negative_vs_cf',
    column='NDCGDifference',
)
add_cases(
    failure_base, 'largest_positive_vs_cf',
    column='NDCGDifference', ascending=False,
)
failure_case_audit = pd.concat(cases, ignore_index=True).drop_duplicates([
    'AuditCategory', 'learner_id'
])
audit_columns = [
    'AuditCategory', 'learner_id', 'Route', 'EarlyInteractions',
    'HasSupportedSkillEvidence', 'RelevantCount', 'RecommendedCount',
    'RecallAtK', 'NDCGAtK', 'RecallDifference', 'NDCGDifference',
    'IntraListDiversity', 'MeanNoveltyBits',
    'CFContribution', 'SequentialContribution', 'ContentContribution',
    'RecommendedItems', 'HitPositions',
]
failure_case_audit[audit_columns].to_parquet(
    OUTPUT_ROOT / 'failure_case_audit.parquet',
    index=False, compression='snappy',
)
display(failure_case_audit[audit_columns].groupby(
    'AuditCategory'
).size().rename('Cases'))

,Cases
AuditCategory,
highest_diversity,10
largest_negative_vs_cf,10
largest_positive_vs_cf,10
lowest_diversity,10
no_hit,10
popularity_fallback,10
sequential_route,4
short_list,10


## Step 12 — Create report-ready figures and source tables


In [16]:
plot_core = ranking_metrics[
    ranking_metrics['CandidatePolicy'].eq('all_supported')
    & ranking_metrics['RelevanceDefinition'].eq('attempted')
    & ranking_metrics['Segment'].eq('all')
][['Model', 'K', 'RecallAtK', 'NDCGAtK']]
plot_core.to_csv(OUTPUT_ROOT / 'figure_core_metrics.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for model, group in plot_core.groupby('Model'):
    axes[0].plot(group['K'], group['RecallAtK'], marker='o', label=model)
    axes[1].plot(group['K'], group['NDCGAtK'], marker='o', label=model)
axes[0].set(title='Recall by K', xlabel='K', ylabel='Recall')
axes[1].set(title='NDCG by K', xlabel='K', ylabel='NDCG')
axes[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'ranking_metrics_by_k.png', dpi=180)
plt.close(fig)

plot_extended = (
    diversity_metrics[
        diversity_metrics['CandidatePolicy'].eq('all_supported')
        & diversity_metrics['RelevanceDefinition'].eq('attempted')
        & diversity_metrics['Segment'].eq('all')
    ][['K', 'MeanIntraListDiversity']]
    .merge(
        novelty_metrics[
            novelty_metrics['CandidatePolicy'].eq('all_supported')
            & novelty_metrics['RelevanceDefinition'].eq('attempted')
            & novelty_metrics['Segment'].eq('all')
        ][['K', 'MeanNoveltyBits']],
        on='K', validate='one_to_one',
    )
)
plot_extended.to_csv(
    OUTPUT_ROOT / 'figure_diversity_novelty.csv', index=False
)
fig, axis = plt.subplots(figsize=(6, 4))
axis.plot(
    plot_extended['K'], plot_extended['MeanIntraListDiversity'],
    marker='o', label='Diversity',
)
second = axis.twinx()
second.plot(
    plot_extended['K'], plot_extended['MeanNoveltyBits'],
    marker='s', color='tab:orange', label='Novelty bits',
)
axis.set(xlabel='K', ylabel='Intra-list diversity')
second.set_ylabel('Novelty bits')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'diversity_novelty_by_k.png', dpi=180)
plt.close(fig)

plot_routes = route_metrics[
    route_metrics['CandidatePolicy'].eq('all_supported')
    & route_metrics['RelevanceDefinition'].eq('attempted')
][['Route', 'Learners', 'RecallAt10', 'NDCGAt10']]
plot_routes.to_csv(OUTPUT_ROOT / 'figure_route_metrics.csv', index=False)
fig, axis = plt.subplots(figsize=(7, 4))
axis.bar(plot_routes['Route'], plot_routes['Learners'])
axis.set(title='Learners by recommendation route', ylabel='Learners')
axis.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'route_usage.png', dpi=180)
plt.close(fig)

plot_subgroups = subgroup_metrics.copy()
plot_subgroups.to_csv(
    OUTPUT_ROOT / 'figure_subgroup_metrics.csv', index=False
)
stable = plot_subgroups[plot_subgroups['StableSummary']]
fig, axis = plt.subplots(figsize=(10, 4))
labels = stable['Dimension'] + ': ' + stable['Group']
axis.bar(labels, stable['NDCGAt10'])
axis.set(title='NDCG@10 by stable early-evidence subgroup', ylabel='NDCG@10')
axis.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'subgroup_ndcg.png', dpi=180)
plt.close(fig)

paired[['learner_id', 'RecallDifference', 'NDCGDifference']].to_csv(
    OUTPUT_ROOT / 'figure_paired_differences.csv', index=False
)
fig, axis = plt.subplots(figsize=(7, 4))
axis.hist(paired['NDCGDifference'], bins=40)
axis.axvline(0, color='black', linewidth=1)
axis.set(
    title='Learner-level NDCG@10 difference',
    xlabel='Promoted minus neighbour CF', ylabel='Learners',
)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'paired_ndcg_distribution.png', dpi=180)
plt.close(fig)

## Step 13 — Save the Phase 5 results


In [17]:
primary_row = paired_uncertainty[
    paired_uncertainty['Metric'].eq('NDCGAt10')
].iloc[0]
evaluation_decision = pd.DataFrame([{
    'ProductionModel': PRODUCTION_MODEL,
    'MeanNDCGDifferenceVsCF': primary_row['Mean'],
    'NDCGLower95': primary_row['Lower95'],
    'NDCGUpper95': primary_row['Upper95'],
}])
evaluation_decision.to_csv(
    OUTPUT_ROOT / 'evaluation_decision.csv', index=False
)

phase5_config = {
    'phase': 5,
    'implemented_steps': list(range(1, 14)),
    'production_model': PRODUCTION_MODEL,
    'primary_comparator': PRIMARY_COMPARATOR,
    'ks': list(KS),
    'bootstrap_resamples': BOOTSTRAP_RESAMPLES,
    'personalization_pair_sample': PAIR_SAMPLE_SIZE,
    'random_state': RANDOM_STATE,
    'diversity_definition': 'one_minus_association_weighted_skill_jaccard',
    'diversity_metadata_fallback': 'problem_type_and_hierarchy_jaccard',
    'novelty_definition': 'negative_log2_discovery_early_popularity',
    'novel_relevance_excludes_seen': True,
    'subgroups_use_early_or_frozen_data_only': True,
    'validation_scope': 'frozen_cohort_post_selection',
}
with open(
    OUTPUT_ROOT / 'phase5_config.json', 'w', encoding='utf-8'
) as file:
    json.dump(phase5_config, file, indent=2)
display(evaluation_decision)

,ProductionModel,MeanNDCGDifferenceVsCF,NDCGLower95,NDCGUpper95
0,gated_cf_content_with_sequential_fallback,0.000353,-0.000048,0.000868
